# Can we understand the Letterboxd algorithm?

Letterboxd, like most sites that allow users to rate items, has a rating algorithm that takes all the users' ratings and gives out an "average". 
Perharps, we would think that the average is just the mean of all the ratings. But, as we saw on the EDA, users tend to have biases that skew the ratings. 
For example, some users might be supporters of certain performer, genre or director and rate those movies higher. 

In this notebook, we will try to understand the ways the Letterboxd algorithm corrects biases. We will use the paper, "Behind the Stars: Uncovering Hidden Adjustments in Letterboxd Film Ratings", by Santana Trigueiro et al (Conference: Proceedings of the Brazilian Symposium on Multimedia and the Web, 2025).

# Loading the movies metadata dataset

In [25]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ky1338/10000-movies-letterboxd-data")

print("Path to dataset files:", path)


Path to dataset files: /Users/virginiapedreira/.cache/kagglehub/datasets/ky1338/10000-movies-letterboxd-data/versions/1


In [26]:
import pandas as pd
import os

# See what files are in there
print(os.listdir(path))



['Movie_Data_File.csv']


In [27]:
df_movies = pd.read_csv(os.path.join(path, 'Movie_Data_File.csv'))


# Quick look at the data

In [46]:
df_movies.head()

,Film_title,Release_year,Director,Cast,Average_rating,Genres,Runtime,Countries,Original_language,Spoken_languages,...,★★,★★½,★★★,★★★½,★★★★,★★★★½,★★★★★,Total_ratings,Film_URL,star_sum
0,The Fan,NaN,Eckhart Schmidt,"['Désirée Nosbusch', 'Bodo Staiger', 'Simone B...",3.57,"['Horror', 'Drama']",92.0,['Germany'],German,['German'],...,402,525,1660,1950,2646,808,714,9042,https://letterboxd.com/film/the-fan-1982/,9042
1,Mad Max: Fury Road,NaN,George Miller,"['Tom Hardy', 'Charlize Theron', 'Nicholas Hou...",4.18,"['Adventure', 'Science Fiction', 'Action']",121.0,"['Australia', 'USA']",English,['English'],...,37471,30112,158356,163753,477901,280815,511140,1682389,https://letterboxd.com/film/mad-max-fury-road/,1682389
2,Suspiria,NaN,Dario Argento,"['Jessica Harper', 'Stefania Casini', 'Flavio ...",3.93,['Horror'],99.0,['Italy'],English,"['English', 'French', 'German', 'Italian', 'La...",...,11006,14397,53427,70309,138742,60628,88628,443757,https://letterboxd.com/film/suspiria/,443757
3,Lost in Translation,NaN,Sofia Coppola,"['Bill Murray', 'Scarlett Johansson', 'Akiko T...",3.79,"['Drama', 'Comedy', 'Romance']",102.0,"['UK', 'USA']",English,"['English', 'Japanese']",...,45997,46716,155110,166638,314160,122359,193717,1076949,https://letterboxd.com/film/lost-in-translation/,1076949
4,Akira,NaN,Katsuhiro Otomo,"['Mitsuo Iwata', 'Nozomu Sasaki', 'Mami Koyama...",4.28,"['Animation', 'Action', 'Science Fiction']",124.0,['Japan'],Japanese,['Japanese'],...,7286,9544,40850,61104,168485,112657,196532,600721,https://letterboxd.com/film/akira/,600721


In [47]:
print(df_movies.columns.tolist())
print(df_movies.shape)
print(df_movies.dtypes)

['Film_title', 'Release_year', 'Director', 'Cast', 'Average_rating', 'Genres', 'Runtime', 'Countries', 'Original_language', 'Spoken_languages', 'Description', 'Studios', 'Watches', 'List_appearances', 'Likes', 'Fans', '½', '★', '★½', '★★', '★★½', '★★★', '★★★½', '★★★★', '★★★★½', '★★★★★', 'Total_ratings', 'Film_URL', 'star_sum']
(10002, 29)
Film_title               str
Release_year         float64
Director                 str
Cast                     str
Average_rating       float64
Genres                   str
Runtime              float64
Countries                str
Original_language        str
Spoken_languages         str
Description              str
Studios                  str
Watches                int64
List_appearances       int64
Likes                  int64
Fans                   int64
½                      int64
★                      int64
★½                     int64
★★                     int64
★★½                    int64
★★★                    int64
★★★½                 

Trying to understand if the Owner rating column refers to the ratings of the person that scraped the website. We are going to compare them with the first ten average ratings and also count them. 

In [48]:
print(df_movies[['Film_title', 'Average_rating', 'Owner_rating']].head(10))
print(df_movies["Owner_rating"].isna().sum())

KeyError: "['Owner_rating'] not in index"

As we can see, the owner rating column only has perfect ratings from the list [0.5, 1, 1.5, ...]. Also, there are 9035 NaNs in that colums. This means that this is the data of one individual, most likely the scraper. We're going to remove that column. 

In [ ]:
df_movies = df_movies.drop(columns=['Owner_rating'])
print(df_movies.shape)

(10002, 28)


Now, let's check if the average rating is the mathematical average or the Letterboxd rating.

In [ ]:
print(df_movies.iloc[:10, 17: 28])
print(df_movies.iloc[0, 27])

       ★     ★½     ★★    ★★½     ★★★    ★★★½    ★★★★   ★★★★½   ★★★★★  \
0    129    103    402    525    1660    1950    2646     808     714   
1  12530   6139  37471  30112  158356  163753  477901  280815  511140   
2   2814   2710  11006  14397   53427   70309  138742   60628   88628   
3  15167  11281  45997  46716  155110  166638  314160  122359  193717   
4   1822   1663   7286   9544   40850   61104  168485  112657  196532   
5     90    141    595   1067    4274    6148    7868    2484    1677   
6   1250   1435   6083   8842   30582   38642   53871   17844   24938   
7   6041   2936  21993  22650  161969  184400  570725  256761  550396   
8    351    379   1635   2615   11549   19412   46693   30296   39520   
9   1410   1202   6036   8236   36987   52496   85867   26402   18997   

   Total_ratings                                           Film_URL  
0           9042          https://letterboxd.com/film/the-fan-1982/  
1        1682389     https://letterboxd.com/film/mad-max

In [ ]:
print(df_movies.shape)

(10002, 28)


In [ ]:
star_cols = ['½', '★', '★½', '★★', '★★½', '★★★', '★★★½', '★★★★', '★★★★½', '★★★★★']
df_movies['star_sum'] = df_movies[star_cols].sum(axis=1)

print(df_movies[['Film_title', 'star_sum', 'Total_ratings']].head(10))

                    Film_title  star_sum  Total_ratings
0                      The Fan      9042           9042
1           Mad Max: Fury Road   1682389        1682389
2                     Suspiria    443757         443757
3          Lost in Translation   1076949        1076949
4                        Akira    600721         600721
5               Things to Come     24383          24383
6  Big Trouble in Little China    183927         183927
7                    Star Wars   1780203        1780203
8                The Third Man    152601         152601
9                 The Big Sick    238309         238309


We will use the fact that we have the number of each type of ratings (1/2 a star, 1 star, ...) to calculate the mathematical average of the first ten movies. This is to understand the meaning of the columns. 

In [ ]:
math_average = []
total_ratings =[]
for i in range(0,10):
    average = 0
    sum = 0
    for n in range(1,11):
        average = average + df_movies.iloc[i , 15 + n] * 0.5 * n
        sum = sum + df_movies.iloc[i, 15 + n]
    math_average.append(float(average / sum))
    total_ratings.append(sum)

print(math_average)
print(df_movies[['Average_rating', 'Total_ratings']].head(10))
print(total_ratings)


[3.544293297942933, 4.132945174986284, 3.9272124158041453, 3.7775029272509655, 4.233426166223588, 3.6727843169421317, 3.726252806820097, 4.175834722219881, 4.155251276203956, 3.726976320659312]
   Average_rating  Total_ratings
0            3.57           9042
1            4.18        1682389
2            3.93         443757
3            3.79        1076949
4            4.28         600721
5            3.77          24383
6            3.72         183927
7            4.17        1780203
8            4.25         152601
9            3.76         238309
[np.int64(9042), np.int64(1682389), np.int64(443757), np.int64(1076949), np.int64(600721), np.int64(24383), np.int64(183927), np.int64(1780203), np.int64(152601), np.int64(238309)]


We can tell that that the number of total ratings calculated by adding hte number of ratings with each star coincides with the total number of ratings. Also, the average rating is similar to the average rating from the database but not so close. The average rating of the first movie for example is 3.5442 and the math average is 3.57, not really a rounding error. I think we can assume that the Average rating is the Letterboxd rating. 

# Merge the ratings data set with the movies dataset

Now we will try to merge our two datasets. Problem is that our ratings datasets (with many users ratings of movies) has a movie id but not the movies dataset. However, the good thing is that the movie title in the ratings dataset corresponds with the Letterboxd slugs (which are the part of the URL that identifies a page in a website, for example now-you-see-me-now-you-dont-2025 is the slug and the url is https://letterboxd.com/film/now-you-see-me-now-you-dont-2025/). The movies database stores the urls of the movies pages so we will connect the two databases through that. 

In [57]:
# Import the dataset with ratings 
df_ratings = pd.read_parquet("../data/raw/ratings.parquet")

# Extract slug from URL
df_movies['slug'] = df_movies['Film_URL'].str.extract(r'/film/([^/]+)/')

print(df_movies['slug'].head(10))
print(df_ratings['title'].head(10))

0                   the-fan-1982
1              mad-max-fury-road
2                       suspiria
3            lost-in-translation
4                          akira
5            things-to-come-2016
6    big-trouble-in-little-china
7                      star-wars
8                  the-third-man
9                   the-big-sick
Name: slug, dtype: str
0                                      scream-7
1                                    ella-mccay
2                                    zootopia-2
3                               wicked-for-good
4              now-you-see-me-now-you-dont-2025
5                             predator-badlands
6                                     tron-ares
7    taylor-swift-the-official-release-party-of
8                                 black-phone-2
9                  a-big-bold-beautiful-journey
Name: title, dtype: str


Let's see if we can merge them. We will create a new dataframe merged, with the movies that are in both databases. We will count how many of the movies are matched and how many movies are in each dataset. 

In [ ]:
df_merged = df_movies.merge(df_ratings, left_on='slug', right_on='title', how='inner')
print(f"Matched movies: {df_merged['slug'].nunique()}")
print(f"Total in df_movies: {len(df_movies)}")
print(f"Total in df_ratings: {df_ratings['title'].nunique()}")

In [ ]:
print(df_merged.shape)
print(df_merged.head(10))

(5195459, 34)
  Film_title  Release_year         Director  \
0    The Fan           NaN  Eckhart Schmidt   
1    The Fan           NaN  Eckhart Schmidt   
2    The Fan           NaN  Eckhart Schmidt   
3    The Fan           NaN  Eckhart Schmidt   
4    The Fan           NaN  Eckhart Schmidt   
5    The Fan           NaN  Eckhart Schmidt   
6    The Fan           NaN  Eckhart Schmidt   
7    The Fan           NaN  Eckhart Schmidt   
8    The Fan           NaN  Eckhart Schmidt   
9    The Fan           NaN  Eckhart Schmidt   

                                                Cast  Average_rating  \
0  ['Désirée Nosbusch', 'Bodo Staiger', 'Simone B...            3.57   
1  ['Désirée Nosbusch', 'Bodo Staiger', 'Simone B...            3.57   
2  ['Désirée Nosbusch', 'Bodo Staiger', 'Simone B...            3.57   
3  ['Désirée Nosbusch', 'Bodo Staiger', 'Simone B...            3.57   
4  ['Désirée Nosbusch', 'Bodo Staiger', 'Simone B...            3.57   
5  ['Désirée Nosbusch', 'Bodo Staige

In [ ]:
del df_merged

We don't need this dataset since we are going to be using mostly the movies metadata (the Letterboxd rating and a mathematical average) but not individual ratigns, so I deleted that dataframe to save space. 